# Análise exploratória — Cartões no Brasileirão Série A (2020–2023)

**Autor:** Natan Braslavsky · Bacharelado em Sistemas de Informação · 6º período
**Disciplina:** Linguagem de Programação (Análise e Visualização de Dados com Python)

Este notebook documenta a exploração preliminar, a prototipação das métricas e a validação do pipeline. As funções reutilizáveis ficam em `src/etl.py`, `src/metricas.py` e `src/figuras.py`; aqui elas são chamadas e os resultados intermediários são inspecionados.

**Perguntas de pesquisa**

- **RQ1.** Quanto o volume de cartões por partida varia entre árbitros, e quais árbitros se afastam do padrão da liga além do esperado pelo acaso?
- **RQ2.** A presença de torcida está associada a um viés disciplinar contra o time visitante?
- **RQ3.** Em que momentos da partida os cartões se concentram, e como a taxa de punição muda conforme o placar momentâneo?

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import json
import numpy as np
import pandas as pd
import plotly.io as pio

from src import etl, metricas as m, figuras as fg

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pio.renderers.default = "notebook_connected"


## 1. Inspeção do arquivo bruto

O CSV é lido inteiramente como texto (`dtype=str`) para que nenhuma conversão implícita aconteça antes de ser auditada.

In [2]:
bruto = etl.carregar_bruto()
print(bruto.shape)
bruto.head(3)

(1520, 17)


,home_team,away_team,home_team_score,away_team_score,game_date,stadium,public,ref,yellow_cards_home,red_cards_home,gols_home,yellow_cards_away,red_cards_away,gols_away,sec_card_home,sec_card_away,link
0,CR Vasco da Gama,Goiás EC,3,2,25 de fevereiro de 2021 21:30,São Januário,No data,Leandro Pedro Vuaden,"[{""player"": ""Leonardo de Matos Cruz"", ""minute""...",[],"[{""player"": ""Germ\u00e1n Ezequiel Cano"", ""minu...","[{""player"": ""Miguel Ferreira Damasceno"", ""minu...",[],"[{""player"": ""Jos\u00e9 Fernando Viana de Santa...",[],[],https://optaplayerstats.statsperform.com/pt_BR...
1,São Paulo FC,CR Flamengo,2,1,25 de fevereiro de 2021 21:30,Morumbi,No data,Rodolpho Toski Marques,"[{""player"": ""Danilo das Neves Pinheiro"", ""minu...",[],"[{""player"": ""Luciano da Rocha Neves"", ""minute""...","[{""player"": ""Gabriel Barbosa Almeida"", ""minute...",[],"[{""player"": ""Bruno Henrique Pinto"", ""minute"": ...",[],[],https://optaplayerstats.statsperform.com/pt_BR...
2,SC Internacional,SC Corinthians Paulista,0,0,25 de fevereiro de 2021 21:30,Beira-Rio,No data,Wilton Pereira Sampaio,[],[],[],"[{""player"": ""V\u00edctor Danilo Cantillo Jim\u...",[],[],[],[],https://optaplayerstats.statsperform.com/pt_BR...


In [3]:
print("Valores textuais em 'public':", (bruto["public"] == "No data").sum())
print("Exemplos de público:", bruto.loc[bruto["public"] != "No data", "public"].head(6).tolist())
print("Exemplos de data:", bruto["game_date"].sample(4, random_state=1).tolist())

Valores textuais em 'public': 613
Exemplos de público: ['4.215', '12.444', '5.304', '17.056', '33.577', '51.125']
Exemplos de data: ['11 de maio de 2023 19:00', '12 de agosto de 2020 21:30', '3 de dezembro de 2023 18:30', '13 de junho de 2022 19:00']


Três armadilhas de *parsing* aparecem logo na inspeção:

1. **Público com separador de milhar em ponto** (`"12.089"`). Um `pd.to_numeric` ingênuo leria 12,089 torcedores em vez de 12 089.
2. **Datas por extenso em português** (`"25 de fevereiro de 2021 21:30"`), que `pd.to_datetime` não interpreta.
3. **Colunas com listas JSON** (cartões e gols, com jogador e minuto, incluindo acréscimos no formato `"90+3'"`).

Além disso, a temporada não é a do ano da data: a temporada 2020 terminou em fevereiro de 2021. A temporada é extraída do link da fonte.

In [4]:
ingenuo = pd.to_numeric(bruto["public"], errors="coerce")
correto = bruto["public"].map(etl.converter_publico)
pd.DataFrame({"texto": bruto["public"], "to_numeric": ingenuo, "convertido": correto}).dropna().head(5)

,texto,to_numeric,convertido
380,4.215,4.215,4215.0
381,12.444,12.444,12444.0
382,5.304,5.304,5304.0
383,17.056,17.056,17056.0
384,33.577,33.577,33577.0


In [5]:
pd.DataFrame({
    "game_date": bruto["game_date"].head(3),
    "convertido": bruto["game_date"].head(3).map(etl.converter_data),
    "ano_da_data": bruto["game_date"].head(3).map(etl.converter_data).dt.year,
    "temporada_link": bruto["link"].head(3).map(etl.extrair_temporada),
})

,game_date,convertido,ano_da_data,temporada_link
0,25 de fevereiro de 2021 21:30,2021-02-25 21:30:00,2021,2020
1,25 de fevereiro de 2021 21:30,2021-02-25 21:30:00,2021,2020
2,25 de fevereiro de 2021 21:30,2021-02-25 21:30:00,2021,2020


## 2. Execução do pipeline e relatório de rastreabilidade

In [6]:
partidas, eventos = etl.executar_pipeline()
relatorio = json.loads((etl.PROCESSED_DIR / "relatorio_etl.json").read_text(encoding="utf-8"))
pd.json_normalize(relatorio)

,etapa,linhas,colunas,datas_invalidas,temporadas_invalidas,publico_ausente,publico_valor_textual_no_data,removidas,partida_ids,divergentes,por_temporada.2020,por_temporada.2021,por_temporada.2022,por_temporada.2023,por_torcida.com_publico,por_torcida.portoes_fechados,por_torcida.desconhecido,em_acrescimo,por_tipo.amarelo,por_tipo.gol,por_tipo.vermelho,por_tipo.segundo_amarelo,arquivos
0,leitura,1520.0,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,conversao_tipos,NaN,NaN,0.0,0.0,613.0,613.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,remocao_duplicatas,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,descarte_linhas_invalidas,NaN,NaN,NaN,NaN,NaN,NaN,0.0,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,checagem_placar_vs_lista_gols,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,partidas_final,1520.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,380.0,380.0,380.0,380.0,907.0,598.0,15.0,NaN,NaN,NaN,NaN,NaN,NaN
6,eventos_final,11558.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1922.0,7474.0,3637.0,298.0,149.0,NaN
7,persistencia,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"[data/processed/partidas.parquet, data/process..."


In [7]:
partidas.info()

<class 'pandas.DataFrame'>
RangeIndex: 1520 entries, 0 to 1519
Data columns (total 30 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   partida_id                   1520 non-null   str           
 1   temporada                    1520 non-null   Int16         
 2   data_hora                    1520 non-null   datetime64[us]
 3   mandante                     1520 non-null   category      
 4   visitante                    1520 non-null   category      
 5   mandante_nome_oficial        1520 non-null   str           
 6   visitante_nome_oficial       1520 non-null   str           
 7   estadio                      1520 non-null   category      
 8   arbitro                      1520 non-null   category      
 9   publico                      907 non-null    float64       
 10  torcida                      1520 non-null   category      
 11  gols_mandante                1520 non-null   Int8     

In [8]:
eventos.head(10)

,partida_id,temporada,tipo,lado,time,jogador,minuto_texto,minuto_base,acrescimo,gol_contra,penalti,tempo,minuto_ordenacao,em_acrescimo,placar_mandante_antes,placar_visitante_antes,saldo_time_antes,situacao_placar
0,2voa0xg8wbeekqp7a079mkp3o,2022,gol,mandante,Atlético-MG,Givanildo Vieira de Souza,10',10,0,False,False,1,10.00,False,0,0,0,empatando
1,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,visitante,Internacional,Keiller da Silva Nunes,37',37,0,<NA>,<NA>,1,37.00,False,1,0,-1,perdendo
2,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,visitante,Internacional,Gabriel Girotto Franco,45',45,0,<NA>,<NA>,1,45.00,False,1,0,-1,perdendo
3,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,mandante,Atlético-MG,Nathanael Ananias da Silva,55',55,0,<NA>,<NA>,2,55.00,False,1,0,1,vencendo
4,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,mandante,Atlético-MG,Guilherme Antonio Arana Lopes,62',62,0,<NA>,<NA>,2,62.00,False,1,0,1,vencendo
5,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,visitante,Internacional,Carlos María de Pena Bonino,68',68,0,<NA>,<NA>,2,68.00,False,1,0,-1,perdendo
6,2voa0xg8wbeekqp7a079mkp3o,2022,amarelo,visitante,Internacional,Gabriel Iván Mercado,77',77,0,<NA>,<NA>,2,77.00,False,1,0,-1,perdendo
7,2voa0xg8wbeekqp7a079mkp3o,2022,gol,mandante,Atlético-MG,Givanildo Vieira de Souza,90+3',90,3,False,False,2,90.03,True,1,0,1,vencendo
8,2vuokv7vsr3d6muinb8a1zswk,2022,amarelo,mandante,Fluminense,Manoel Messias Silva Carvalho,48',48,0,<NA>,<NA>,2,48.00,False,0,0,0,empatando
9,2vzat1qfkyyzcy67mhqphxzbo,2022,gol,mandante,São Paulo,Jonathan Calleri,19',19,0,False,False,1,19.00,False,0,0,0,empatando


### Validações de integridade

- O número de gols nas listas JSON bate com o placar em todas as partidas.
- Nenhum jogador aparece ao mesmo tempo em `red_cards` e `sec_card` na mesma partida: o vermelho direto e a expulsão por segundo amarelo são registros distintos.
- Cada temporada tem exatamente 380 partidas (20 clubes, turno e returno).

In [9]:
assert (partidas.groupby("temporada").size() == 380).all()
assert partidas["partida_id"].is_unique
gols = eventos[eventos["tipo"] == "gol"].groupby(["partida_id", "lado"], observed=True).size().unstack(fill_value=0)
conferencia = partidas.set_index("partida_id")[["gols_mandante", "gols_visitante"]].join(gols).fillna(0)
assert (conferencia["gols_mandante"] == conferencia["mandante"]).all()
assert (conferencia["gols_visitante"] == conferencia["visitante"]).all()
print("validações ok")

validações ok


## 3. Estatística descritiva

In [10]:
pd.Series(m.resumo_descritivo(partidas["cartoes_total"])).round(3)

n                   1520.000
media                  5.211
mediana                5.000
desvio_padrao          2.579
variancia              6.653
coef_variacao          0.495
q1                     3.000
q3                     7.000
iqr                    4.000
minimo                 0.000
maximo                16.000
assimetria             0.554
indice_dispersao       1.277
dtype: float64

In [11]:
partidas[["cartoes_total", "amarelos_mandante", "amarelos_visitante", "expulsoes_total", "publico"]].describe().round(2)

,cartoes_total,amarelos_mandante,amarelos_visitante,expulsoes_total,publico
count,1520.00,1520.00,1520.00,1520.00,907.00
mean,5.21,2.44,2.58,0.29,22908.94
std,2.58,1.52,1.59,0.61,16220.95
min,0.00,0.00,0.00,0.00,79.00
25%,3.00,1.00,1.00,0.00,9157.00
50%,5.00,2.00,2.00,0.00,19621.00
75%,7.00,3.00,4.00,0.00,35815.50
max,16.00,8.00,11.00,5.00,69997.00


O índice de dispersão (variância/média ≈ 1,28) indica **sobredispersão** em relação a um processo de Poisson: há mais partidas com poucos e com muitos cartões do que o esperado se todas as partidas tivessem a mesma propensão. Essa heterogeneidade é justamente o que as RQs investigam (árbitro, torcida e contexto do jogo).

In [12]:
fg.fig_distribuicao_cartoes(partidas)

In [13]:
outliers = partidas[m.outliers_iqr(partidas["cartoes_total"])]
outliers[["temporada", "mandante", "visitante", "arbitro", "cartoes_total", "expulsoes_total"]]

,temporada,mandante,visitante,arbitro,cartoes_total,expulsoes_total
695,2021,Fluminense,Palmeiras,Caio Max Augusto Vieira,14,3
762,2022,Palmeiras,Ceará,Caio Max Augusto Vieira,16,1
887,2022,Internacional,Botafogo,Sávio Pereira Sampaio,16,5
1269,2023,Goiás,Coritiba,Denis Ribeiro Serafim,14,0
1282,2023,Fluminense,Flamengo,Sávio Pereira Sampaio,15,1
1336,2023,Santos,Grêmio,Paulo Cesar Zanovelli da Silva,14,2


Os seis *outliers* pelo critério de Tukey (1,5 × IQR, acima de 13 cartões) são partidas reais e plausíveis, em geral com expulsões e jogos tensos (como o Fla-Flu de 2023). Eles foram **mantidos** na base e apenas sinalizados, já que removê-los distorceria justamente a cauda que interessa.

In [14]:
fg.fig_correlacao(partidas)

## 4. RQ1 — Variação entre árbitros

In [15]:
perfil = m.perfil_arbitros(partidas)
perfil[perfil["elegivel"]][["arbitro", "jogos", "cartoes_por_jogo", "expulsoes_por_jogo", "classificacao"]].round(2)

,arbitro,jogos,cartoes_por_jogo,expulsoes_por_jogo,classificacao
8,Rodrigo José Pereira de Lima,28,7.07,0.50,acima_do_esperado
12,Paulo Cesar Zanovelli da Silva,43,6.56,0.51,acima_do_esperado
13,Rafael Rodrigo Klein,24,6.50,0.25,dentro_do_esperado
15,Braulio da Silva Machado,90,6.47,0.36,acima_do_esperado
18,Bruno Arleu de Araújo,66,5.98,0.30,dentro_do_esperado
20,Felipe Fernandes de Lima,32,5.78,0.25,dentro_do_esperado
22,Luiz Flávio de Oliveira,57,5.75,0.39,dentro_do_esperado
24,Wagner do Nascimento Magalhaes,60,5.72,0.22,dentro_do_esperado
26,Caio Max Augusto Vieira,44,5.57,0.32,dentro_do_esperado
28,Wilton Pereira Sampaio,71,5.49,0.28,dentro_do_esperado


In [16]:
m.teste_diferenca_arbitros(partidas)

{'arbitros': 25,
 'partidas': 1277,
 'H': np.float64(138.45433224565187),
 'p_valor': np.float64(4.474124493834105e-18),
 'epsilon2': np.float64(0.09141719827927465)}

In [17]:
fg.fig_funil_arbitros(partidas)

## 5. RQ2 — Torcida e viés contra o visitante

In [18]:
m.vies_mandante_por_torcida(partidas).round(3).T

,0,1
torcida,com_publico,portoes_fechados
partidas,907,598
amarelos_mandante,2.521,2.299
amarelos_visitante,2.796,2.264
dif_media,0.275,-0.035
ic95_inf,0.149,-0.181
ic95_sup,0.399,0.11
expulsoes_mandante,0.137,0.129
expulsoes_visitante,0.164,0.147
vitorias_mandante,0.476,0.425


In [19]:
m.teste_vies_torcida(partidas)

{'diferenca_das_medias': np.float64(0.30964847912741106),
 'ic95_inf': 0.12103188504128058,
 'ic95_sup': 0.5016755133797701,
 'mann_whitney_U': np.float64(296293.0),
 'p_valor_mann_whitney': np.float64(0.002039210302668073),
 'p_valor_wilcoxon_fechados': np.float64(0.5088897516255955),
 'p_valor_wilcoxon_publico': np.float64(3.629386937918213e-05)}

In [20]:
fg.fig_temporadas(partidas)

In [21]:
m.vies_por_faixa_publico(partidas).round(3)

,faixa_publico,partidas,dif_media,ic95_inf,ic95_sup
0,Portões fechados,598,-0.035,-0.181,0.110
1,Até 10 mil,250,0.080,-0.144,0.308
2,10 a 20 mil,212,0.297,0.042,0.552
3,20 a 35 mil,210,0.310,0.033,0.576
4,Acima de 35 mil,235,0.430,0.179,0.681


In [22]:
fg.fig_faixa_publico(partidas)

In [23]:
m.correlacao_publico_vies(partidas)

{'partidas': 907,
 'rho_spearman': np.float64(0.06838755212618312),
 'p_valor': np.float64(0.03947886511681114)}

## 6. RQ3 — Momento da partida e placar

In [24]:
m.concentracao_segundo_tempo(eventos)

{'cartoes': 7921,
 'fracao_segundo_tempo': np.float64(0.6502966797121575),
 'fracao_acrescimos': np.float64(0.18570887514202752),
 'minuto_mediano': np.float64(62.0)}

In [25]:
m.distribuicao_minutos(eventos)

tipo,faixa,amarelo,segundo_amarelo,vermelho
0,1–15,333,1,14
1,16–30,793,2,19
2,31–45,1171,12,26
3,45+ (acréscimos),385,2,12
4,46–60,1001,22,29
5,61–75,1289,37,39
6,76–90,1566,32,64
7,90+ (acréscimos),936,41,95


In [26]:
fg.fig_minutos(eventos)

Contar cartões por situação de placar sem considerar quanto tempo cada time passou em cada situação gera um viés: os times passam muito mais tempo empatando no início do jogo, quando há poucos cartões. Por isso a métrica é uma **taxa por 90 minutos de exposição**, estratificada por tempo de jogo.

In [27]:
taxas = m.taxa_cartoes_por_situacao(partidas, eventos, ("tempo", "lado"))
taxas.round(3)

,tempo,lado,situacao_placar,minutos,cartoes,cartoes_por_90,ic95_inf,ic95_sup
0,1,mandante,perdendo,8564.68,247,2.596,2.272,2.919
1,1,mandante,empatando,47608.86,748,1.414,1.313,1.515
2,1,mandante,vencendo,13746.46,342,2.239,2.002,2.476
3,1,visitante,perdendo,13746.46,370,2.422,2.176,2.669
4,1,visitante,empatando,47608.86,810,1.531,1.426,1.637
5,1,visitante,vencendo,8564.68,253,2.659,2.331,2.986
6,2,mandante,perdendo,15712.44,633,3.626,3.343,3.908
7,2,mandante,empatando,24495.59,812,2.983,2.778,3.189
8,2,mandante,vencendo,26711.67,1061,3.575,3.360,3.790
9,2,visitante,perdendo,26711.67,994,3.349,3.141,3.557


In [28]:
por_tempo = m.taxa_cartoes_por_situacao(partidas, eventos, ("tempo",))
{t: m.teste_taxas_situacao(por_tempo[por_tempo["tempo"] == t]) for t in (1, 2)}

{1: {'qui2': np.float64(179.41954102130626),
  'p_valor': np.float64(1.0953217066153802e-39),
  'taxa_perdendo': np.float64(2.488891199642869),
  'taxa_empatando': np.float64(1.4726250534039251),
  'taxa_vencendo': np.float64(2.4001462946312917),
  'razao_perdendo_vencendo': np.float64(1.0369747899159665)},
 2: {'qui2': np.float64(25.258114830988106),
  'p_valor': np.float64(3.2754430502951276e-06),
  'taxa_perdendo': np.float64(3.451575059559293),
  'taxa_empatando': np.float64(3.1964937362194585),
  'taxa_vencendo': np.float64(3.7846403849132013),
  'razao_perdendo_vencendo': np.float64(0.9119955156950673)}}

In [29]:
fg.fig_taxa_situacao(partidas, eventos)

## 7. Exportação dos artefatos do artigo

In [30]:
_ = fg.gerar_figuras()
sorted(p.name for p in fg.DIR_FIGURAS.iterdir())

['fig-arbitros.png',
 'fig-correlacao.png',
 'fig-distribuicao.png',
 'fig-minutos.png',
 'fig-publico.png',
 'fig-situacao.png',
 'fig-temporadas.png']